# Experiment 1 baseline: California ACS 2023
Same standard MLP, 10 predictors, split sizes, seed, early stopping,
four attack features and random forest attack as Experiment 1.
The relationship field uses native-year codes (RELP in 2014; RELSHIPP in 2023).
Income eligibility and nominal >$50,000 label follow ACSIncome; no inflation adjustment.
Auxiliary source: 2014; 16 diverse shadows.
Target and auxiliary preprocessing fit their own training records only.
The user authorized this run; ask before any future experiment run.
Census documentation: https://www.census.gov/programs-surveys/acs/microdata/documentation.html

This notebook includes target training, the 16-shadow membership attack,
and saved-model tests on 25,000 records each from 2024 and 2018.


In [1]:
from pathlib import Path
import json
import hashlib
import importlib.metadata
import joblib
import pandas as pd
import numpy as np
import torch
torch.set_num_threads(4)
from src.data import split_acs_data, save_partition_ids
from src.recent_data import load_recent_acs, preprocess_recent_partitions
from src.training import train_target_model, evaluate_model, save_target_model

# Open this notebook from the project root. Use a new RUN_DIR per experiment.
SEED = 42
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=SEED)
TRAIN_CONFIG = dict(hidden_dims=(128, 64), dropout=0.0, epochs=50,
                    batch_size=256, learning_rate=1e-3, weight_decay=1e-4,
                    patience=5, seed=SEED, device='cpu')
TARGET_YEAR = 2023
AUXILIARY_SOURCE = '2014'
N_SHADOW_MODELS = 16
RUN_DIR = Path('artifacts/experiment1_2023_aux2014_16shadows')


## Load and split
IDs refer to rows of this exact CSV, not unique people. Identical feature vectors may occur naturally. Keep test and MIA partitions out of model selection.

In [2]:
data = load_recent_acs(TARGET_YEAR)
partitions = split_acs_data(data, **SPLIT_CONFIG)
pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
              for name, part in partitions.items()}).T

,records,positive_rate
validation,10000.0,0.498400
test,10000.0,0.498400
mia_nonmembers,10000.0,0.498400
train,173278.0,0.498448
mia_members,10000.0,0.498400


## Preprocess
Age and weekly hours are standardized. Other Census codes, including education, are one-hot encoded. Missing values are imputed using training statistics; unseen categories are ignored. The original partition frames remain available for later feature masking.

In [3]:
preprocessor, arrays = preprocess_recent_partitions(partitions)
{name: values['X'].shape for name, values in arrays.items()}

{'validation': (10000, 819),
 'test': (10000, 819),
 'mia_nonmembers': (10000, 819),
 'train': (173278, 819),
 'mia_members': (10000, 819)}

## Train
This cell trains the standard MLP and restores the epoch with the lowest validation loss. It does not train a differentially private model.

In [4]:
model, history, metadata = train_target_model(
    arrays['train']['X'], arrays['train']['y'],
    arrays['validation']['X'], arrays['validation']['y'],
    **TRAIN_CONFIG,
)
pd.DataFrame(history)
RUN_DIR.mkdir(parents=True, exist_ok=True)
save_target_model(model, metadata, RUN_DIR / 'target_model.pt')
joblib.dump(preprocessor, RUN_DIR / 'preprocessor.joblib')
save_partition_ids(partitions, RUN_DIR)


Epoch 001: train loss=0.4189, validation loss=0.4013, validation accuracy=0.8195


Epoch 002: train loss=0.3942, validation loss=0.3951, validation accuracy=0.8194


Epoch 003: train loss=0.3877, validation loss=0.3938, validation accuracy=0.8220


Epoch 004: train loss=0.3830, validation loss=0.3959, validation accuracy=0.8207


Epoch 005: train loss=0.3782, validation loss=0.3965, validation accuracy=0.8199


Epoch 006: train loss=0.3731, validation loss=0.3939, validation accuracy=0.8178


Epoch 007: train loss=0.3686, validation loss=0.3953, validation accuracy=0.8192


Epoch 008: train loss=0.3640, validation loss=0.3966, validation accuracy=0.8184


## Final predictive evaluation
Run after fixing hyperparameters using validation only. These are income-prediction metrics, not membership-attack metrics. Do not tune the model using these results.

In [5]:
metrics = {name: evaluate_model(model, arrays[name]['X'], arrays[name]['y'])
           for name in ['train', 'validation', 'test']}
pd.DataFrame(metrics).T

,accuracy,roc_auc,log_loss
train,0.82943,0.911237,0.377457
validation,0.82200,0.902280,0.393789
test,0.81900,0.904673,0.390328


## Random forest membership attack: complete-feature baseline
Run after training the original 2023 target. This uses the same attack procedure as the overfit notebook: 16 ACS 2014 auxiliary shadows, four paper-inspired features (class-scaled logits, blind walk, entropy, cross-entropy), and a random forest membership classifier.

The attack is fitted on ACS 2014 auxiliary shadow outputs only. ACS 2014 auxiliary attack rows are split by record ID; thresholds are selected on ACS 2014 auxiliary validation only. Feature extraction samples 2,000 members and 2,000 non-members per shadow. Target evaluation uses all 10,000 original target MIA members and 10,000 non-members, with this target's own fitted preprocessor. No overfit results are reused or compared.

Walk defaults: 20 steps, two walks, categorical change probability .05, batch size 512, seed 42. Repeated model queries can take time. The tabular walk is an adaptation, not an exact reproduction; see data/paper_attack_features.md. True income labels are assumed known. No target MIA results select settings. All cells were executed with user authorization for this run.

In [6]:
import importlib
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features

WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000
# Train the 16 ACS 2014 auxiliary shadows when you run this cell.
# Skip this cell if shadow_results is already loaded and these imports/settings are set.
adult_auxiliary = load_recent_acs(2014)
shadow_results = train_shadow_models(adult_auxiliary, seed=42, schema='acs', n_models=N_SHADOW_MODELS)
summarize_shadow_models(shadow_results)
from src.paper_attack_features import make_probability_oracle
for name, result in shadow_results.items():
    rows = result['attack_data'].iloc[:32]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema=result['schema'])
    np.testing.assert_allclose(query(adult_auxiliary.loc[rows['record_id']]),
        rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(), rtol=1e-5, atol=1e-6)
print('Probability consistency verified for all', N_SHADOW_MODELS, 'shadows.')


Training shadow 1/16: xgboost


Training shadow 2/16: random_forest


Training shadow 3/16: mlp


Epoch 001: train loss=0.4143, validation loss=0.3776, validation accuracy=0.8258


Epoch 002: train loss=0.3691, validation loss=0.3672, validation accuracy=0.8314


Epoch 003: train loss=0.3603, validation loss=0.3642, validation accuracy=0.8332


... (31 outputs left out)

Training shadow 14/16: decision_tree


Training shadow 15/16: logistic_regression


Training shadow 16/16: xgboost


Probability consistency verified for all 16 shadows.


In [7]:
# Extract the four features by querying this notebook's fitted shadows.
adult_for_rf_queries = adult_auxiliary
shadow_attack_data, rf_shadow_walk_diagnostics = collect_shadow_paper_features(
    shadow_results, adult_for_rf_queries, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=42)
display(pd.DataFrame(rf_shadow_walk_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=42, algorithm='random_forest', feature_set='paper')
# ACS 2014 auxiliary attack diagnostics, not target membership performance.
pd.DataFrame(attack_result['metrics']).T

Attack feature extraction: 512/4,000


Attack feature extraction: 1,024/4,000


Attack feature extraction: 1,536/4,000


Attack feature extraction: 2,048/4,000


Attack feature extraction: 2,560/4,000


Attack feature extraction: 3,072/4,000


... (120 outputs left out)

Attack feature extraction: 3,584/4,000


Attack feature extraction: 4,000/4,000


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
xgboost_01,124544,164000,0.62775,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_02,128613,164000,0.661125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_03,123561,164000,0.61325,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_04,111253,164000,0.50625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_05,125577,164000,0.61425,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,125713,164000,0.6355,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,128123,164000,0.657375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp_08,124787,164000,0.620875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree_09,113873,164000,0.537875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression_10,125051,164000,0.610625,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38419.0,19317.0,0.590932,0.590989,0.628617,0.023451,0.504869,11484.0,7618.0,8098.0,11219.0,0.583234,0.015220,0.005340
validation,12849.0,6376.0,0.517939,0.517875,0.520037,0.010822,0.504869,3407.0,3066.0,3128.0,3248.0,0.583234,0.010822,0.007724
test,12732.0,6307.0,0.502356,0.502324,0.506791,0.010465,0.504869,3250.0,3175.0,3161.0,3146.0,0.583234,0.010465,0.009027


In [8]:
# Query this notebook's original target using its own fitted preprocessor.
target_attack_data, rf_target_walk_diagnostics = collect_target_paper_features(
    model, preprocessor, partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(rf_target_walk_diagnostics, name='target query diagnostics'))
target_attack_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
pd.DataFrame({
    'ACS 2014 auxiliary attack test': attack_result['metrics']['test'],
    'ACS 2023 target MIA': target_attack_metrics,
}).T

Attack feature extraction: 512/20,000


Attack feature extraction: 1,024/20,000


Attack feature extraction: 1,536/20,000


Attack feature extraction: 2,048/20,000


Attack feature extraction: 2,560/20,000


Attack feature extraction: 3,072/20,000


... (32 outputs left out)

Attack feature extraction: 19,968/20,000


Attack feature extraction: 20,000/20,000


record_queries                                                       593101
max_record_queries                                                   820000
walk_censored_fraction                                              0.54155
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
ACS 2014 auxiliary attack test,12732.0,6307.0,0.502356,0.502324,0.506791,0.010465,0.504869,3250.0,3175.0,3161.0,3146.0,0.583234,0.010465,0.009027
ACS 2023 target MIA,20000.0,10000.0,0.501750,0.501750,0.504306,0.011100,0.504869,5088.0,4912.0,5053.0,4947.0,0.583234,0.009300,0.009000


In [9]:

RUN_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history).to_csv(RUN_DIR / 'target_history.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(RUN_DIR / 'shadow_metrics.csv')
joblib.dump(attack_result['classifier'], RUN_DIR / 'attack_classifier.joblib')
summary = dict(target_year=TARGET_YEAR, auxiliary_source=AUXILIARY_SOURCE,
    n_shadow_models=len(shadow_results), original_features=list(preprocessor.feature_names_in_),
    encoded_features=arrays['train']['X'].shape[1], split_config=SPLIT_CONFIG,
    partition_counts={name:len(part) for name,part in partitions.items()}, training=metadata,
    target_metrics=metrics, target_mia_metrics=target_attack_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_query_diagnostics=rf_target_walk_diagnostics,
    shadow_query_diagnostics=rf_shadow_walk_diagnostics,
    oracle_consistency_verified=True,
    source_sha256=hashlib.file_digest(Path(f'data/raw/folktables/{TARGET_YEAR}/1-Year/csv_pca.zip').open('rb'), 'sha256').hexdigest())
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Results saved to', RUN_DIR)


Results saved to artifacts\experiment1_2023_aux2014_16shadows


In [1]:
# Test the saved 2023 target on 25,000 ACS 2024 records (no retraining).
# Fixed stratified sample; fitted target preprocessor is loaded unchanged.
# 2018 RELP is converted to RELSHIPP with a reported mapping audit.
# These are income-prediction metrics; keep them out of model selection.
import importlib
from pathlib import Path
import pandas as pd
from IPython.display import display
import src.recent_data
importlib.reload(src.recent_data)
from src.recent_data import evaluate_saved_target_external

external_2024_metrics, external_2024_coverage = evaluate_saved_target_external(
    2024, Path('artifacts/experiment1_2023_aux2014_16shadows'), sample_size=25_000, seed=42)
display(pd.DataFrame([external_2024_metrics]).set_index('evaluation_year'))
display(external_2024_coverage)


,target_year,records,eligible_source_records,seed,sampling,positive_rate,accuracy,roc_auc,log_loss,relationship_mapping_audit,relationship_audit_scope,notes
evaluation_year,,,,,,,,,,,,
2024,2023,25000,203591,42,"stratified by income label, without replacement",0.52076,0.81328,0.898314,0.403231,None,all eligible source records,"Nominal >$50,000 income; no inflation adjustme..."


,unseen_records,unseen_fraction
feature,,
COW,0,0.00000
SCHL,0,0.00000
MAR,0,0.00000
OCCP,1,0.00004
POBP,0,0.00000
RELSHIPP,0,0.00000
SEX,0,0.00000
RAC1P,0,0.00000


In [2]:
# Test the saved 2023 target on 25,000 ACS 2018 records (no retraining).
# Fixed stratified sample; fitted target preprocessor is loaded unchanged.
# 2018 RELP is converted to RELSHIPP with a reported mapping audit.
# These are income-prediction metrics; keep them out of model selection.
import importlib
from pathlib import Path
import pandas as pd
from IPython.display import display
import src.recent_data
importlib.reload(src.recent_data)
from src.recent_data import evaluate_saved_target_external

external_2018_metrics, external_2018_coverage = evaluate_saved_target_external(
    2018, Path('artifacts/experiment1_2023_aux2014_16shadows'), sample_size=25_000, seed=42)
display(pd.DataFrame([external_2018_metrics]).set_index('evaluation_year'))
display(external_2018_coverage)


,target_year,records,eligible_source_records,seed,sampling,positive_rate,accuracy,roc_auc,log_loss,relationship_mapping_audit,relationship_audit_scope,notes
evaluation_year,,,,,,,,,,,,
2018,2023,25000,195665,42,"stratified by income label, without replacement",0.41056,0.81048,0.903067,0.408811,"{'direct_category_mapping': 143894, 'spouse_pa...",all eligible source records,"Nominal >$50,000 income; no inflation adjustme..."


,unseen_records,unseen_fraction
feature,,
COW,0,0.0
SCHL,0,0.0
MAR,0,0.0
OCCP,0,0.0
POBP,0,0.0
RELSHIPP,0,0.0
SEX,0,0.0
RAC1P,0,0.0
